# 🖐️ YOLOv8n Hand Detection — Full Training Pipeline

**What this notebook does:**
1. Installs all required libraries
2. Mounts Google Drive (saves your work permanently)
3. Downloads a free pre-labeled hand dataset from Roboflow
4. Trains YOLOv8n with transfer learning
5. Exports the model to ONNX format
6. Tests inference on your laptop

---
> ⚠️ **IMPORTANT — Before running anything:**
> - You must use **Google Colab** (https://colab.research.google.com)
> - Make sure to **enable GPU**: Runtime → Change runtime type → T4 GPU
> - Run cells **one by one from top to bottom**
> - If your session disconnects, just re-run from the top (Drive keeps your data)


---
## STEP 1 — Install Required Libraries

Run this cell first. It installs everything needed.  
⏳ Takes about 1–2 minutes. You will see a lot of output — that is normal.


In [ ]:
# Install all required packages
# The ! prefix means: run this as a terminal command inside Colab
!pip install ultralytics roboflow onnx onnxruntime opencv-python --quiet

# Verify GPU is available
import torch
print("=" * 50)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✅ GPU detected: {gpu_name}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ No GPU detected!")
    print("   Go to: Runtime → Change runtime type → Hardware accelerator → T4 GPU")
    print("   Then re-run this cell.")
print("=" * 50)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.0/184.0 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 107.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 57.6 MB/s eta 0:00:00
✅ GPU detected: Tesla T4
   VRAM: 15.6 GB


---
## STEP 2 — Mount Google Drive

This saves ALL your files (dataset, trained model) to your Google Drive.  
So if Colab disconnects, you won't lose anything.

**What happens when you run this cell:**
1. A link appears → click it
2. Choose your Google account
3. Click "Allow"
4. Copy the code → paste it in the box → press Enter


In [ ]:
from google.colab import drive
import os

# Mount Google Drive at /content/drive
drive.mount('/content/drive')

# Create our project folders inside Google Drive
PROJECT_DIR = "/content/drive/MyDrive/hand_detection"
DATASET_DIR = f"{PROJECT_DIR}/dataset"
RUNS_DIR    = f"{PROJECT_DIR}/runs"
MODELS_DIR  = f"{PROJECT_DIR}/models"

for folder in [DATASET_DIR, RUNS_DIR, MODELS_DIR]:
    os.makedirs(folder, exist_ok=True)

print("✅ Google Drive mounted!")
print(f"   Project folder: {PROJECT_DIR}")
print(f"   All your files will be saved here automatically.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted!
   Project folder: /content/drive/MyDrive/hand_detection
   All your files will be saved here automatically.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---
## ⚡ RESTORE PATHS — Run this after every reconnect

> If Colab disconnects and you restart the session, you **must** re-run:
> - Cell 3 (Install libraries)
> - Cell 5 (Mount Drive)
> - **This cell** (restore path variables)
>
> Then you can jump directly to whichever step you need (train / eval / export).


In [ ]:
import os

# ── All project paths defined in one place ──────────────────
PROJECT_DIR = "/content/drive/MyDrive/hand_detection"
DATASET_DIR = f"{PROJECT_DIR}/dataset"
RUNS_DIR    = f"{PROJECT_DIR}/runs"
MODELS_DIR  = f"{PROJECT_DIR}/models"

# Path to dataset config (exists after Step 3 has run once)
DATA_YAML   = f"{DATASET_DIR}/egohands-public-9/data.yaml"

# Path to best trained weights (exists after Step 4 has run once)
BEST_PT     = f"{RUNS_DIR}/hand_yolov8n/weights/best.pt"

# Verify what already exists on Drive
print("📁 Path status:")
for label, path in [("PROJECT_DIR", PROJECT_DIR),
                     ("DATASET_DIR", DATASET_DIR),
                     ("DATA_YAML  ", DATA_YAML),
                     ("BEST_PT    ", BEST_PT)]:
    status = "✅" if os.path.exists(path) else "⬜ (not yet created)"
    print(f"   {label}: {status}")
    if label == "PROJECT_DIR":
        os.makedirs(MODELS_DIR, exist_ok=True)


📁 Path status:
   PROJECT_DIR: ✅
   DATASET_DIR: ✅
   DATA_YAML  : ⬜ (not yet created)
   BEST_PT    : ⬜ (not yet created)


---
## STEP 3 — Download Hand Dataset from Roboflow (Free)

We use a **free public dataset** from Roboflow Universe.  
It has ~3600 images already labeled — no manual work needed!

**How to get your free API key (takes 2 minutes):**
1. Go to https://app.roboflow.com → Sign up free
2. After login → click your **avatar (top right)** → **Settings**
3. Click **"Roboflow API"** tab → copy your **Private API Key**
4. Paste it below where it says `YOUR_API_KEY_HERE`


In [ ]:
from roboflow import Roboflow
import os

# ──────────────────────────────────────────────────────────
# ⚠️  PASTE YOUR FREE API KEY HERE (from app.roboflow.com)
ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"
# ──────────────────────────────────────────────────────────

if ROBOFLOW_API_KEY == "YOUR_API_KEY_HERE":
    raise ValueError("❌ Please replace YOUR_API_KEY_HERE with your actual Roboflow API key!")

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Dataset: EgoHands — 4800 images, exactly 1 class: "hand"
# Much better than gesture datasets for pure hand bounding box detection
# Source: https://universe.roboflow.com/brad-dwyer/egohands-public
print("📥 Downloading EgoHands dataset from Roboflow Universe...")
project = rf.workspace("brad-dwyer").project("egohands-public")
dataset = project.version(9).download("yolov8", location=DATASET_DIR, overwrite=True)

# Update the global DATA_YAML variable with actual downloaded path
DATA_YAML = dataset.location + "/data.yaml"

print(f"\n✅ Dataset downloaded!")
print(f"   Location : {dataset.location}")
print(f"   data.yaml: {DATA_YAML}")

# Quick stats
for split in ["train", "valid", "test"]:
    img_dir = os.path.join(dataset.location, split, "images")
    if os.path.exists(img_dir):
        n = len(os.listdir(img_dir))
        print(f"   {split:6s}: {n} images")

print(f"\n💾 DATA_YAML path saved. You can now run Step 4 (Training).")


📥 Downloading EgoHands dataset from Roboflow Universe...
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/drive/MyDrive/hand_detection/dataset in yolov8:: 100%|██████████| 24426/24426 [05:22<00:00, 75.81it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

✅ Dataset downloaded!
   Location : /content/drive/MyDrive/hand_detection/dataset
   data.yaml: /content/drive/MyDrive/hand_detection/dataset/data.yaml
   train : 11247 images
   valid : 480 images
   test  : 480 images

💾 DATA_YAML path saved. You can now run Step 4 (Training).


---
## STEP 4 — Train YOLOv8n with Transfer Learning

Training takes about **15–30 minutes** on Colab free T4 GPU.

**What transfer learning means here:**
- `yolov8n.pt` = pretrained on COCO dataset (knows 80 general object classes)
- We re-train the last layers to detect only hands
- Much faster and needs less data than training from scratch

**Parameters explained:**
| Parameter | Value | Why |
|-----------|-------|-----|
| `epochs` | 50 | Number of full passes through the dataset |
| `imgsz` | 320 | Image size (smaller = faster, better for RPi later) |
| `batch` | 16 | Images per training step (16 fits T4 GPU) |
| `patience` | 15 | Auto-stop if no improvement after 15 epochs |


In [ ]:
from ultralytics import YOLO
import os

# Load YOLOv8n pretrained on COCO (downloads ~6MB automatically)
model = YOLO("yolov8n.pt")

print("🚀 Starting training...")
print(f"   Dataset  : {DATA_YAML}")
print(f"   Results  : {RUNS_DIR}")
print()

results = model.train(
    data      = DATA_YAML,        # path to dataset config
    epochs    = 50,               # train for up to 50 epochs
    imgsz     = 320,              # image size (320 is good for RPi)
    batch     = 16,               # batch size for T4 GPU
    lr0       = 0.01,             # initial learning rate
    patience  = 15,               # early stopping patience
    device    = 0,                # 0 = use GPU
    project   = RUNS_DIR,         # save results to Google Drive
    name      = "hand_yolov8n",   # subfolder name
    exist_ok  = True,             # overwrite if re-running
    verbose   = True,
)

# Path to the best trained weights
BEST_PT = f"{RUNS_DIR}/hand_yolov8n/weights/best.pt"
print(f"\n✅ Training complete!")
print(f"   Best model saved: {BEST_PT}")
print(f"   mAP50 : {results.results_dict.get('metrics/mAP50(B)', 'N/A'):.3f}")


🚀 Starting training...
   Dataset  : /content/drive/MyDrive/hand_detection/dataset/data.yaml
   Results  : /content/drive/MyDrive/hand_detection/runs

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/hand_detection/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=trai

---
## STEP 5 — Evaluate the Model

Check the accuracy metrics. You want:
- **mAP50 > 0.80** = good model ✅
- **mAP50 < 0.60** = need more data or more epochs ⚠️


In [ ]:
from ultralytics import YOLO

# Load the best trained model
model = YOLO(BEST_PT)

# Evaluate on validation set
metrics = model.val(data=DATA_YAML, imgsz=320)

print("\n📊 Evaluation Results:")
print(f"   mAP50     : {metrics.box.map50:.3f}   (target > 0.80)")
print(f"   mAP50-95  : {metrics.box.map:.3f}   (target > 0.50)")
print(f"   Precision : {metrics.box.mp:.3f}")
print(f"   Recall    : {metrics.box.mr:.3f}")

if metrics.box.map50 >= 0.80:
    print("\n✅ Great! Model is ready for export.")
elif metrics.box.map50 >= 0.65:
    print("\n⚠️  Acceptable. Consider training more epochs if time allows.")
else:
    print("\n❌ Low accuracy. Try: epochs=100, or check your dataset quality.")


Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.9±0.4 ms, read: 8.2±1.5 MB/s, size: 18.0 KB)
val: Scanning /content/drive/MyDrive/hand_detection/dataset/valid/labels.cache... 480 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 480/480 134.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 30/30 4.4it/s 6.9s
                   all        480       1515      0.987       0.97      0.991      0.795
Speed: 0.8ms preprocess, 2.1ms inference, 0.0ms loss, 2.3ms postprocess per image
Results saved to /content/runs/detect/val

📊 Evaluation Results:
   mAP50     : 0.991   (target > 0.80)
   mAP50-95  : 0.795   (target > 0.50)
   Precision : 0.987
   Recall    : 0.970

✅ Great! Model is ready for export.


---
## STEP 6 — Export to ONNX Format

Converts `best.pt` (PyTorch) → `best.onnx` (ONNX).  
The ONNX file is what you will run on your **laptop and Raspberry Pi** using `onnxruntime`.

After this cell:
1. The file `best.onnx` will be saved in your Google Drive
2. Download it: open the left sidebar in Colab → Files → navigate to `/content/drive/MyDrive/hand_detection/models/`
3. Right-click `best.onnx` → Download


In [ ]:
import shutil
from ultralytics import YOLO

model = YOLO(BEST_PT)

# Export to ONNX (CPU-compatible, works on RPi and laptop)
print("📦 Exporting to ONNX...")
exported_path = model.export(
    format  = "onnx",
    imgsz   = 320,    # must match training imgsz
    opset   = 12,     # opset 12 = widely supported
    dynamic = False,  # fixed input size, better for embedded
)

# Copy exported .onnx to our models folder in Drive
ONNX_DEST = f"{MODELS_DIR}/best.onnx"
shutil.copy(exported_path, ONNX_DEST)

print(f"\n✅ Export complete!")
print(f"   ONNX model saved to Google Drive: {ONNX_DEST}")
print(f"\n📥 To download:")
print(f"   Left sidebar → Files icon → navigate to:")
print(f"   /content/drive/MyDrive/hand_detection/models/best.onnx")
print(f"   Right-click → Download")


📦 Exporting to ONNX...
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/hand_detection/runs/hand_yolov8n/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 5, 2100) (5.9 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 319ms
Prepared 3 packages in 3.20s
Installed 3 packages in 55ms
 + colorama==0.4.6
 + onnxruntime-gpu==1.25.0
 + onnxslim==0.1.91

requirements: AutoUpdate success ✅ 4.1s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.21.0 opset 12...
